In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import shutil
import sys
import gymnasium as gym
import torch

# Ensure repository root is on sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if PROJECT_ROOT not in sys.path:
  sys.path.insert(0, PROJECT_ROOT)

from src.rl_transformer.env_adapter import MatchEnv
from src.rl_transformer.pool import PoolOpponentController
from src.rl_transformer.ppo import train_mappo
from src.rl_transformer.transformer_model import TransformerActorCritic

# ── Hardware Performance Flags ──
torch.backends.cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── Regulation Dimensions & Physics ──
TEAM_SIZE = 1
PITCH_W_REG = 1200.0
PITCH_H_REG = 800.0
GOAL_H_REG = 220.0
ROUND_STEPS = 3600
ACTION_REPEAT = 10
NUM_ENVS = 24

# ── Directories & Seed Checkpoints ──
SAVE_DIR_S1_P3 = "models/stage1/phase3"
SAVE_DIR_S2_P1 = "models/stage2/phase1"
POOL_DIR_S2_P1 = os.path.join(SAVE_DIR_S2_P1, "pool")
os.makedirs(POOL_DIR_S2_P1, exist_ok=True)

seed_file = os.path.join(SAVE_DIR_S1_P3, "best_model.pt")
if not os.path.exists(seed_file):
  seed_file = os.path.join(SAVE_DIR_S1_P3, "final_model.pt")

if not os.path.exists(seed_file):
  raise FileNotFoundError(f"Missing Stage 1 Phase 3 checkpoint: {seed_file}")

shutil.copy(seed_file, os.path.join(POOL_DIR_S2_P1, "champion.pt"))
shutil.copy(seed_file, os.path.join(POOL_DIR_S2_P1, "history_0.pt"))
print(f"🔥 Stage 2 Phase 1 seeded from: {seed_file}")


# ── Environment Factory ──
def make_s2_p1_env(env_rank: int):
  def _thunk():
    torch.set_num_threads(1)

    opp_ctrl = PoolOpponentController(
        pool_dir=POOL_DIR_S2_P1,
        team="blue",
        device="cpu",
        p_random=0.05,  
        p_heuristic=0.35,  
        frame_stack=3,
    )
    env = MatchEnv(
        team_size=TEAM_SIZE,
        learner_team_size=TEAM_SIZE,
        opp_team_size=TEAM_SIZE,
        learner_team="red",
        max_round_steps=ROUND_STEPS,
        action_repeat=ACTION_REPEAT,
        goal_height=GOAL_H_REG,
        pitch_width=PITCH_W_REG,
        pitch_height=PITCH_H_REG,
        opponent_controller=opp_ctrl,
        opponent_stats=[
            (3200.0, 1200.0),  # Standard tier
            (3600.0, 1400.0),  # Buffed Tier 1
            (4000.0, 1600.0),  # Buffed Tier 2 (extreme speed/kick)
        ],
        frame_stack=3,
    )
    env.reset(seed=4000 + env_rank)
    return env

  return _thunk


envs_s2_p1 = gym.vector.AsyncVectorEnv(
    [make_s2_p1_env(i) for i in range(NUM_ENVS)],
    context="fork",
    shared_memory=False,
)

# ── Model Initialization & Warmstart ──
model_s2_p1 = TransformerActorCritic().to(device)
ckpt = torch.load(seed_file, map_location=device, weights_only=False)
state_dict = (
    ckpt["model_state_dict"]
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt
    else ckpt
)
model_s2_p1.load_state_dict(state_dict, strict=True)
print("✅ Weights successfully loaded into Stage 2 Phase 1 model.")

# ── Training Loop ──
train_mappo(
    envs=envs_s2_p1,
    model=model_s2_p1,
    device=device,
    team_size=TEAM_SIZE,
    opp_team_size=TEAM_SIZE,
    total_timesteps=50_000_000,
    num_envs=NUM_ENVS,
    num_steps=512,  # Batch = 16 * 512 = 8,192
    update_epochs=3,
    minibatch_size=2048,
    lr_init=3e-5,
    lr_final=3e-6,
    ent_coef_init=0.01,
    ent_coef_final=0.001,
    gamma=0.997,
    gae_lambda=0.96,
    active_tiers=["heuristic", "champion"],
    target_tier="champion",
    filter_thresholds={"heuristic": 0.90},
    tier_ratios={"heuristic": 0.50, "champion": 0.50},
    eval_episodes=100,
    eval_freq=250_000,
    save_dir=SAVE_DIR_S2_P1,
    pool_dir=POOL_DIR_S2_P1,
    goal_height=GOAL_H_REG,
    pitch_width=PITCH_W_REG,
    pitch_height=PITCH_H_REG,
    max_steps=ROUND_STEPS,
    action_repeat=ACTION_REPEAT,
)

envs_s2_p1.close()

Using device: cuda
🔥 Stage 2 Phase 1 seeded from: models/stage1/phase3/best_model.pt
✅ Weights successfully loaded into Stage 2 Phase 1 model.
🚀 Entity-Transformer MAPPO Initialized | Format: 1v1 | Envs: 24 | Batch: 12288 | Device: cuda

📊 [EVALUATION @ Step 258,048 | Rollout SPS: 2033 | Tiers: ['heuristic', 'champion']]
   ⚔️  vs Heuristic [FILTER] | WR:  96.0% | Reward: +4.978 | Goals: 195 Scored, 11 Conceded (+184 Net)
   ⚔️  vs Champion  [TARGET] | WR:  26.0% | Reward: -0.458 | Goals: 43 Scored, 52 Conceded (-9 Net)
   ❌ Retaining current baseline. Did not pass criteria for champion: [WR: 26.0%, Reward: -0.458, Net: -9] (Eval took 21.2s)

📊 [EVALUATION @ Step 503,808 | Rollout SPS: 2040 | Tiers: ['heuristic', 'champion']]
   ⚔️  vs Heuristic [FILTER] | WR:  98.0% | Reward: +4.952 | Goals: 187 Scored, 6 Conceded (+181 Net)
   ⚔️  vs Champion  [TARGET] | WR:  34.0% | Reward: -0.102 | Goals: 53 Scored, 49 Conceded (+4 Net)
   ❌ Retaining current baseline. Did not pass criteria for cha

In [ ]:
import os
import torch
from src.rl_transformer.visualization import evaluate_and_generate_html

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

html_path = evaluate_and_generate_html(
    red_agent="models/stage2/phase1/best_model.pt",
    blue_agent="heuristic",
    red_team_size=2,
    blue_team_size=2,
    device=device,
    filename="stage2_heuristic_test.html",
    num_episodes=10,
    max_steps=3600,
    action_repeat=4,
    pitch_width=1200.0,
    pitch_height=800.0,
    goal_height=220.0,
)